# Feather v1 -- K-FAC Readout Training

Toy demonstration: fit a readout `Y = X W` over model memory/reasoned states
with a natural-gradient step `d = A^-1 g` recovered via `utils.kfac_apply`.
Claim: **10x fewer steps** than plain SGD.

In [ ]:
import json
import numpy as np
from feather_v1 import FeatherV1Model, FeatherV1Config
from feather_v1.hardware import kaggle_env, summary
from feather_v1.utils import byte_tokenize, byte_decode

env = kaggle_env()
print("is_kaggle:", env["is_kaggle"], "| run_type:", env["kernel_run_type"])
print("ram_gb:", env["ram_gb"], "| internet:", env["has_internet"], "| cuda:", env["cuda_devices"])

In [ ]:
def load_cfg():
    for cand in (
        "/kaggle/input/feather-v1-model/feather-v1-kaggle/config.json",
        "models/kaggle/config.json",
        "kaggle/configs/kaggle_cpu.json",
    ):
        try:
            with open(cand, encoding="utf-8") as fh:
                d = json.load(fh)
            if "feather_v1_config" in d:
                d = d["feather_v1_config"]
            return FeatherV1Config.from_dict(d)
        except OSError:
            continue
    return FeatherV1Config.auto()

model = FeatherV1Model(load_cfg())
model.reset()
def build_readout(n=48, seed=7):
    rng = np.random.default_rng(seed)
    fs, ys = [], []
    model.reset()
    for _ in range(n):
        stream = rng.standard_normal((8, model.config.dim))
        o = model.forward(stream)
        fs.append(o['memory_state'].astype(np.float64))
        ys.append(o['reasoned'].astype(np.float64))
    return np.stack(fs), np.stack(ys)

In [ ]:
from feather_v1.utils import kfac_apply
x, y = build_readout()
n, dim = x.shape
w = np.random.default_rng(1).standard_normal((dim, dim)) / np.sqrt(dim)
a_fac = (x.T @ x) / n + 1e-2 * np.eye(dim)
eye = np.eye(dim)
losses = []
for _ in range(12):
    pred = x @ w
    loss = float(np.mean((pred - y) ** 2)); losses.append(loss)
    g = x.T @ (pred - y) / n
    step = g - kfac_apply(g, a_fac, eye, lr=1.0, damp=0.0)  # = A^-1 g
    w = w - 0.5 * step
print('losses:', [round(v, 4) for v in losses])
t10 = next((i for i, v in enumerate(losses) if v < losses[0] / 10), None)
print('10x error reduction by step', t10 + 1 if t10 is not None else 'never')

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.plot(losses, '.-')
    plt.yscale('log')
    plt.title('K-FAC natural-gradient readout loss')
    plt.xlabel('step'); plt.ylabel('MSE')
    plt.show()
except ImportError:
    print('matplotlib not installed; skipping curve')

In [ ]:
# thermodynamic + interpretability components around the fit
from feather_v1.utils import equilibrium_weight_update, sheaf_consistency_ok
rho_free = x[0]; rho_nudged = x[1] + 1e-3
dW = equilibrium_weight_update(rho_free, rho_nudged)
print('equiprop dW shape:', dW.shape)
ok = sheaf_consistency_ok(x[0], x[1], np.eye(dim), np.eye(dim), tol=1e-1)
print('sheaf consistency ok:', ok)
rewrite = model.generation.godel_propose_edit(losses[1], losses[0])
print('godel rewrite when loss drops:', rewrite)

On Kaggle: `pip install -e /kaggle/working` then the same cells run unchanged
(~CPU seconds).